# 7.24 同一題，怎樣教不同預算的回答？


現在想讓一個小助理接受 `<low>` 或 `<high>`。題目仍是「先有2件，新增3件，再移除1件，剩幾件？」正解都是4；我們沒有為 low 發明一個比較不正確的答案。改變的是**提供給模型的條件**，以及訓練時展示的作答方式。

先沿用[7.11](https://birdhackor.github.io/tiny-perceptron-vlm/7.11.html)的監督式微調（SFT）：把條件與問題當輸入，只有助理示範的回答位置計入 loss。下面兩筆是人工示範資料，程式只展示配對，不更新權重。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
records = [
    {"effort": "low", "question": "先有2件，新增3件，再移除1件，剩幾件？", "answer": "4"},
    {"effort": "high", "question": "先有2件，新增3件，再移除1件，剩幾件？", "answer": "2+3=5；5-1=4。答案4。"},
]
for row in records:
    print(f"<{row['effort']}> {row['question']} -> {row['answer']}")

訓練時，effort 標記也要真的進入模型。若只在資料表中加欄、模型根本沒讀到，就沒有教它依條件改變回答。同一問題不同示範可以用不同 loss mask；但這裡兩份都要學正確答案與結束，不能只對 high 的理由計分，忘了最後答案。

為什麼同一題可放兩次？模型學的是「給定問題與檔位後，這份輸出應有高機率」。low 和 high 是不同條件，不是互相衝突的正解。可是這組資料最直接教到的是**短答與步驟的形式差異**；它沒有保證多給計算就更會解題，更沒有重現商業服務未公開的內部推理訓練。即使 high 很會寫長解釋，也要用新題檢查是否更常答對。

資料還有兩個要守的界線。第一，同一題的兩種版本必須留在同一資料切分；不能 low 放訓練、high 放測試，便稱沒有見過題目。第二，長示範讓模型讀到更多受監督 token；配對題數不等於配對訓練工作量。比較時要交代題數、有效回答 token 與更新步數。

Looped 模型提供另一種做法：同一正解可以在不同終止輪數計 loss，內部狀態不必寫成文字。那是改變計算路徑，和本節的預算條件文字示範可以配合；具體更新見[14.13–14.14](https://birdhackor.github.io/tiny-perceptron-vlm/14.13.html)。

練習把 high 示範改成「2+3=6；6-1=4」。最後4仍對，過程卻錯了；這筆資料會教進哪個錯誤？先檢查每個可驗算步驟，再用[C.1–C.2](https://birdhackor.github.io/tiny-perceptron-vlm/C.1.html)分開看回答形式與過程正確。
